In [1]:
# =========================================
# Cell 1 — Install dependencies
# =========================================
!pip -q install google-genai>=1.0.0 datasets>=2.19.0 pandas>=2.2.0 numpy>=1.26.0 scikit-learn>=1.3.0 tqdm>=4.66.0 matplotlib>=3.8.0


In [2]:
# =========================================
# Cell 2 — Keys (type/paste in)
# =========================================
import os, getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter GEMINI_API_KEY (hidden): ").strip()
assert os.environ.get("GEMINI_API_KEY"), "Missing GEMINI_API_KEY"

hf = getpass.getpass("Enter HF_TOKEN (optional; press Enter to skip): ").strip()
if hf:
    os.environ["HF_TOKEN"] = hf

print("✅ GEMINI_API_KEY set. HF_TOKEN provided?", bool(os.environ.get("HF_TOKEN")))


Enter GEMINI_API_KEY (hidden): ··········
Enter HF_TOKEN (optional; press Enter to skip): ··········
✅ GEMINI_API_KEY set. HF_TOKEN provided? True


In [3]:
# =========================================
# Cell 3 — Load FIQASA dataset (flare-fiqasa) from Hugging Face
# =========================================
from datasets import load_dataset

DATASET_NAME = "TheFinAI/flare-fiqasa"
SPLIT = "test"   # choose: "train", "valid", or "test"

ds = load_dataset(DATASET_NAME, split=SPLIT, token=os.environ.get("HF_TOKEN", None))
print(ds)
print("Columns:", ds.column_names)
print("Example row:\n", ds[0])


README.md:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

data/train-00000-of-00001-d0f9b6513e12e0(…):   0%|          | 0.00/100k [00:00<?, ?B/s]

data/test-00000-of-00001-faca082021057ac(…):   0%|          | 0.00/35.8k [00:00<?, ?B/s]

data/valid-00000-of-00001-36997935dc03cb(…):   0%|          | 0.00/29.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/750 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/235 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/188 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 235
})
Columns: ['id', 'query', 'answer', 'text', 'choices', 'gold']
Example row:
 {'id': 'fiqasa938', 'query': 'What is the sentiment of the following financial post: Positive, Negative, or Neutral?\nText: $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.\nAnswer:', 'answer': 'negative', 'text': '$BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.', 'choices': ['negative', 'positive', 'neutral'], 'gold': 0}


In [4]:
# =========================================
# Cell 4 — Gemini 2.5 Flash: helpers (prompt, parsing, API call)
# =========================================
import re, time
from google import genai
from google.genai import types

MODEL_ID = "gemini-2.5-flash"
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def build_prompt(text: str, choices):
    return (
        "Analyze the sentiment of this financial text.\n"
        "Respond ONLY with one label from the Options.\n\n"
        f"Text: {text}\n"
        f"Options: {', '.join(map(str, choices))}\n"
        "Label:"
    )

def extract_label(model_text: str, choices):
    if not model_text:
        return None
    s = str(model_text).strip().lower()

    # exact match
    for c in choices:
        if s == str(c).strip().lower():
            return str(c)

    # whole-word match
    for c in choices:
        pat = r"\b" + re.escape(str(c).strip().lower()) + r"\b"
        if re.search(pat, s):
            return str(c)

    return None

def _safe_get_text(resp):
    t = getattr(resp, "text", None)
    if t:
        return t

    cands = getattr(resp, "candidates", None) or []
    for c in cands:
        content = getattr(c, "content", None)
        parts = getattr(content, "parts", None) if content else None
        if parts:
            texts = []
            for p in parts:
                tx = getattr(p, "text", None)
                if tx:
                    texts.append(tx)
            if texts:
                return "".join(texts)
    return None

def gemini_predict(prompt: str, model: str = MODEL_ID, max_retries: int = 6, sleep_base: float = 1.0):
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.0,
                    max_output_tokens=16,
                ),
            )

            text_out = _safe_get_text(resp)

            usage = getattr(resp, "usage_metadata", None)
            usage_dict = None
            if usage is not None:
                usage_dict = {
                    "prompt_token_count": getattr(usage, "prompt_token_count", None),
                    "candidates_token_count": getattr(usage, "candidates_token_count", None),
                    "total_token_count": getattr(usage, "total_token_count", None),
                }

            finish_reasons = []
            cands = getattr(resp, "candidates", None) or []
            for c in cands:
                finish_reasons.append(getattr(c, "finish_reason", None))

            meta = {
                "usage": usage_dict,
                "finish_reasons": finish_reasons,
                "has_candidates": bool(cands),
            }
            if text_out is None:
                meta["warning"] = "No text returned by Gemini."

            return text_out, meta

        except Exception as e:
            last_err = str(e)
            time.sleep(sleep_base * (2 ** attempt))

    return None, {"error": last_err}


In [5]:
# =========================================
# Cell 5 — Run evaluation (full split by default) + resume support
# =========================================
import os, json, random, time
from tqdm import tqdm

SEED = 42
MAX_SAMPLES = None          # None = FULL split; set int (e.g., 200) to test
SHUFFLE = True
SLEEP_BETWEEN_CALLS = 0.25  # raise to 0.5–1.0 if rate-limited

random.seed(SEED)

os.makedirs("text_responses", exist_ok=True)
os.makedirs("evaluation", exist_ok=True)

run_tag = f"gemini25flash_{DATASET_NAME.replace('/','_')}_{SPLIT}"
raw_path = f"text_responses/{run_tag}.jsonl"

RESET_RUN = True
if RESET_RUN and os.path.exists(raw_path):
    os.remove(raw_path)
    print("🧹 Deleted existing file:", raw_path)

done = set()
if os.path.exists(raw_path):
    with open(raw_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                done.add(json.loads(line)["row_index"])
            except:
                pass
print("Already completed:", len(done))

idxs = list(range(len(ds)))
if SHUFFLE:
    random.shuffle(idxs)
if MAX_SAMPLES is not None:
    idxs = idxs[:MAX_SAMPLES]

with open(raw_path, "a", encoding="utf-8") as f:
    for i in tqdm(idxs, desc="Evaluating"):
        if i in done:
            continue

        ex = ds[i]
        text = ex["text"]
        choices = list(ex["choices"])
        gold = int(ex["gold"])

        prompt = build_prompt(text, choices)
        model_out, meta = gemini_predict(prompt)

        pred_label = extract_label(model_out, choices)
        pred_idx = choices.index(pred_label) if pred_label in choices else -1

        rec = {
            "row_index": i,
            "id": ex.get("id", None),
            "query": ex.get("query", None),
            "answer": ex.get("answer", None),
            "text": text,
            "choices": choices,
            "gold": gold,
            "model_response_raw": model_out,   # may be None
            "predicted_label": pred_label,
            "predicted_index": pred_idx,
            "correct": (pred_idx == gold),
            "meta": meta,
        }

        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

        if SLEEP_BETWEEN_CALLS:
            time.sleep(SLEEP_BETWEEN_CALLS)

print("✅ Saved raw responses to:", raw_path)


Already completed: 0


Evaluating: 100%|██████████| 235/235 [02:39<00:00,  1.47it/s]

✅ Saved raw responses to: text_responses/gemini25flash_TheFinAI_flare-fiqasa_test.jsonl


In [9]:
# =========================================
# Print output
# =========================================
import os, json
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

# load jsonl
rows = []
with open(raw_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

df = pd.DataFrame(rows)

# coerce types
df["gold"] = pd.to_numeric(df["gold"], errors="coerce").astype(int)
df["predicted_index"] = pd.to_numeric(df["predicted_index"], errors="coerce").fillna(-1).astype(int)

valid_mask = df["predicted_index"] >= 0
labels_sorted = sorted(set(df["gold"].tolist()))

# metrics
acc_all = accuracy_score(df["gold"], df["predicted_index"])
acc_valid = accuracy_score(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"]) if valid_mask.any() else None
f1_weighted = f1_score(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"],
                       average="weighted", labels=labels_sorted, zero_division=0) if valid_mask.any() else None
f1_macro = f1_score(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"],
                    average="macro", labels=labels_sorted, zero_division=0) if valid_mask.any() else None
f1_micro = f1_score(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"],
                    average="micro", labels=labels_sorted, zero_division=0) if valid_mask.any() else None

metrics = {
    "run_tag": run_tag,
    "model_id": MODEL_ID,
    "dataset": DATASET_NAME,
    "split": SPLIT,
    "num_samples": int(len(df)),
    "num_valid_predictions": int(valid_mask.sum()),
    "accuracy_all": float(acc_all),
    "accuracy_valid_only": (float(acc_valid) if acc_valid is not None else None),
    "f1_macro_valid_only": (float(f1_macro) if f1_macro is not None else None),
    "f1_micro_valid_only": (float(f1_micro) if f1_micro is not None else None),
    "f1_weighted_valid_only": (float(f1_weighted) if f1_weighted is not None else None),
}

# save artifacts
os.makedirs("evaluation", exist_ok=True)
metrics_path = f"evaluation/{run_tag}_metrics.json"
preds_path = f"evaluation/{run_tag}_predictions.csv"

with open(metrics_path, "w", encoding="utf-8") as f:
    cm = confusion_matrix(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"], labels=labels_sorted) if valid_mask.any() else None
    json.dump(
        {"metrics": metrics, "confusion_matrix_labels": labels_sorted, "confusion_matrix": (cm.tolist() if cm is not None else None)},
        f, ensure_ascii=False, indent=2
    )

df.to_csv(preds_path, index=False)

print("✅ Metrics saved to:", metrics_path)
print("✅ Predictions saved to:", preds_path)
print()

# print in the same format as your screenshot
print("Classification report (valid predictions only):")
if valid_mask.any():
    print(classification_report(
        df.loc[valid_mask, "gold"],
        df.loc[valid_mask, "predicted_index"],
        labels=labels_sorted,
        zero_division=0
    ))
else:
    print("(No valid predictions to report)")


✅ Metrics saved to: evaluation/gemini25flash_TheFinAI_flare-fiqasa_test_metrics.json
✅ Predictions saved to: evaluation/gemini25flash_TheFinAI_flare-fiqasa_test_predictions.csv

Classification report (valid predictions only):
              precision    recall  f1-score   support

           0       0.75      1.00      0.86         3
           1       0.86      0.67      0.75         9
           2       0.50      0.67      0.57         3

    accuracy                           0.73        15
   macro avg       0.70      0.78      0.73        15
weighted avg       0.76      0.73      0.74        15

